### Import Libraries

In [1]:
import pandas as pd
import numpy as np

### Load and Clean Data

#### Feature Definitions (from `epl-training.csv`)

- **Date** — The date that the match took place  
- **HomeTeam** — The team playing at home  
- **AwayTeam** — The team playing away  
- **FTHG** — Goals scored by the home team at full time  
- **FTAG** — Goals scored by the away team at full time  
- **FTR** — Full-time result *(target variable: H = home win, D = draw, A = away win)*  
- **HTHG** — Goals scored by the home team at half time  
- **HTAG** — Goals scored by the away team at half time  
- **HTR** — Half-time result  
- **Referee** — Name of the referee officiating the match  
- **HS** — Total number of shots by the home team  
- **AS** — Total number of shots by the away team  
- **HST** — Total number of shots on target by the home team  
- **AST** — Total number of shots on target by the away team  
- **HF** — Total number of fouls committed by the home team  
- **AF** — Total number of fouls committed by the away team  
- **HC** — Total number of corners by the home team  
- **AC** — Total number of corners by the away team  
- **HY** — Total number of yellow cards received by the home team  
- **AY** — Total number of yellow cards received by the away team  
- **HR** — Total number of red cards received by the home team  
- **AR** — Total number of red cards received by the away team  

In [2]:
epl_train = pd.read_csv('Data_Files/epl-training.csv')
train_df = epl_train.copy()
train_df = train_df.dropna().reset_index(drop=True)   # remove missing values

In [3]:
team_name_mapping = {
    "Nott'm Forest": 'Nottingham Forest'
}

train_df['HomeTeam'] = train_df['HomeTeam'].replace(team_name_mapping)
train_df['AwayTeam'] = train_df['AwayTeam'].replace(team_name_mapping)

In [4]:
train_df['Date'] = pd.to_datetime(train_df['Date'], dayfirst=True)
train_df = train_df.sort_values(by='Date').reset_index(drop=True)
train_df

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR
0,2000-08-19,Charlton,Man City,4.0,0.0,H,2.0,0.0,H,Rob Harris,...,14.0,4.0,6.0,6.0,13.0,12.0,1.0,2.0,0.0,0.0
1,2000-08-19,Chelsea,West Ham,4.0,2.0,H,1.0,0.0,H,Graham Barber,...,10.0,5.0,7.0,7.0,19.0,14.0,1.0,2.0,0.0,0.0
2,2000-08-19,Coventry,Middlesbrough,1.0,3.0,A,1.0,1.0,D,Barry Knight,...,3.0,9.0,8.0,4.0,15.0,21.0,5.0,3.0,1.0,0.0
3,2000-08-19,Derby,Southampton,2.0,2.0,D,1.0,2.0,A,Andy D'Urso,...,4.0,6.0,5.0,8.0,11.0,13.0,1.0,1.0,0.0,0.0
4,2000-08-19,Leeds,Everton,2.0,0.0,H,2.0,0.0,H,Dermot Gallagher,...,8.0,6.0,6.0,4.0,21.0,20.0,1.0,3.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,Ipswich,West Ham,1.0,3.0,A,0.0,1.0,A,T Robinson,...,4.0,6.0,10.0,9.0,4.0,0.0,1.0,1.0,0.0,0.0
9596,2025-05-25,Fulham,Man City,0.0,2.0,A,0.0,1.0,A,A Madley,...,3.0,5.0,11.0,5.0,1.0,6.0,0.0,0.0,0.0,0.0
9597,2025-05-25,Bournemouth,Leicester,2.0,0.0,H,0.0,0.0,D,L Smith,...,7.0,0.0,19.0,16.0,6.0,1.0,0.0,2.0,0.0,0.0
9598,2025-05-25,Liverpool,Crystal Palace,1.0,1.0,D,0.0,1.0,A,D England,...,3.0,5.0,7.0,10.0,11.0,0.0,1.0,0.0,1.0,0.0


In [5]:
epl_test = pd.read_csv('Data_Files/epl-test.csv')
test_df = epl_test.copy()
test_df['Date'] = pd.to_datetime(test_df['Date'], format='mixed', dayfirst=True)
test_df

,Date,HomeTeam,AwayTeam
0,2026-01-31,Leeds,Arsenal
1,2026-01-31,Liverpool,Newcastle
2,2026-01-31,Tottenham,Man City
3,2026-01-31,Wolves,Bournemouth
4,2026-01-31,Aston Villa,Brentford
5,2026-01-31,Brighton,Everton
6,2026-01-31,Chelsea,West Ham
7,2026-01-31,Man United,Fulham
8,2026-01-31,Sunderland,Burnley
9,2026-01-31,Nottingham Forest,Crystal Palace


### Team Form and Rolling Averages (Last k matches)

In [6]:
def create_team_history(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a unified team history DataFrame from match data.
    
    Combines home and away records into a single timeline per team,
    with goals, shots, and other stats from that team's perspective.
    
    Parameters
    ----------
    df : pd.DataFrame
        Match data with columns: Date, HomeTeam, AwayTeam, FTHG, FTAG, FTR,
        HS, AS, HST, AST, HF, AF, HC, AC
    
    Returns
    -------
    pd.DataFrame
        Team-level history sorted by (Team, Date) with columns:
        Date, Team, GF, GA, FTR, Shots, ShotsOnTarget, Fouls, Corners,
        is_home, GoalDiff, Points
    """
    # Home team perspective
    home = df[["Date", "HomeTeam", "FTHG", "FTAG", "FTR", "HS", "HST", "HF", "HC"]].copy()
    home.rename(columns={
        "HomeTeam": "Team",
        "FTHG": "GF",
        "FTAG": "GA",
        "HS": "Shots",
        "HST": "ShotsOnTarget",
        "HF": "Fouls",
        "HC": "Corners"
    }, inplace=True)
    home["is_home"] = 1

    # Away team perspective
    away = df[["Date", "AwayTeam", "FTHG", "FTAG", "FTR", "AS", "AST", "AF", "AC"]].copy()
    away.rename(columns={
        "AwayTeam": "Team",
        "FTAG": "GF",
        "FTHG": "GA",
        "AS": "Shots",
        "AST": "ShotsOnTarget",
        "AF": "Fouls",
        "AC": "Corners"
    }, inplace=True)
    away["is_home"] = 0

    # Combine and sort
    team_history = (
        pd.concat([home, away], ignore_index=True)
        .sort_values(["Team", "Date"])
        .reset_index(drop=True)
    )

    # Derived columns
    team_history["GoalDiff"] = team_history["GF"] - team_history["GA"]
    team_history["Points"] = (
        (team_history["GF"] > team_history["GA"]).astype(int) * 3
        + (team_history["GF"] == team_history["GA"]).astype(int) * 1
        # 3 points for win, 1 for draw, 0 for loss
        # losses automatically get 0
    )

    return team_history

In [7]:
team_history = create_team_history(train_df)
team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points
0,2000-08-19,Arsenal,0.0,1.0,H,14.0,7.0,21.0,9.0,0,-1.0,0
1,2000-08-21,Arsenal,2.0,0.0,H,17.0,12.0,25.0,10.0,1,2.0,3
2,2000-08-26,Arsenal,5.0,3.0,H,18.0,9.0,12.0,8.0,1,2.0,3
3,2000-09-06,Arsenal,2.0,2.0,D,13.0,5.0,22.0,6.0,0,0.0,1
4,2000-09-09,Arsenal,1.0,1.0,D,18.0,11.0,13.0,10.0,0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3.0,0.0,H,20.0,6.0,4.0,7.0,1,3.0,3
19196,2025-05-02,Wolves,0.0,1.0,H,6.0,1.0,5.0,11.0,0,-1.0,0
19197,2025-05-10,Wolves,0.0,2.0,A,10.0,3.0,7.0,8.0,1,-2.0,0
19198,2025-05-20,Wolves,2.0,4.0,H,12.0,3.0,10.0,8.0,0,-2.0,0


In [8]:
k = 5  # number of previous matches to look back over

# Group by team, because we want each team’s history separately
grp = team_history.groupby("Team", group_keys=False)

In [9]:
# Rolling sums for GF, GA, Points over last k matches (excluding current match)
for col in ["GF", "GA", "Points"]:
    team_history[f"{col}_sum_last_{k}"] = grp[col].apply(
        lambda s: s.shift(1).rolling(window=k, min_periods=1).sum()
    )

team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points,GF_sum_last_5,GA_sum_last_5,Points_sum_last_5
0,2000-08-19,Arsenal,0.0,1.0,H,14.0,7.0,21.0,9.0,0,-1.0,0,NaN,NaN,NaN
1,2000-08-21,Arsenal,2.0,0.0,H,17.0,12.0,25.0,10.0,1,2.0,3,0.0,1.0,0.0
2,2000-08-26,Arsenal,5.0,3.0,H,18.0,9.0,12.0,8.0,1,2.0,3,2.0,1.0,3.0
3,2000-09-06,Arsenal,2.0,2.0,D,13.0,5.0,22.0,6.0,0,0.0,1,7.0,4.0,6.0
4,2000-09-09,Arsenal,1.0,1.0,D,18.0,11.0,13.0,10.0,0,0.0,1,9.0,6.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3.0,0.0,H,20.0,6.0,4.0,7.0,1,3.0,3,10.0,4.0,15.0
19196,2025-05-02,Wolves,0.0,1.0,H,6.0,1.0,5.0,11.0,0,-1.0,0,11.0,3.0,15.0
19197,2025-05-10,Wolves,0.0,2.0,A,10.0,3.0,7.0,8.0,1,-2.0,0,10.0,4.0,12.0
19198,2025-05-20,Wolves,2.0,4.0,H,12.0,3.0,10.0,8.0,0,-2.0,0,8.0,5.0,9.0


In [10]:
# Rolling means for GF, GA, GoalDiff, Shots, ShotsOnTarget, Corners over last k matches (excluding current match)
for col in ["GF", "GA", "GoalDiff", "Shots", "ShotsOnTarget", "Corners"]:
    team_history[f"{col}_mean_last_{k}"] = grp[col].apply(
        lambda s: s.shift(1).rolling(window=k, min_periods=1).mean()
    )

team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,...,Points,GF_sum_last_5,GA_sum_last_5,Points_sum_last_5,GF_mean_last_5,GA_mean_last_5,GoalDiff_mean_last_5,Shots_mean_last_5,ShotsOnTarget_mean_last_5,Corners_mean_last_5
0,2000-08-19,Arsenal,0.0,1.0,H,14.0,7.0,21.0,9.0,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-21,Arsenal,2.0,0.0,H,17.0,12.0,25.0,10.0,1,...,3,0.0,1.0,0.0,0.000000,1.000000,-1.00,14.000000,7.000000,9.00
2,2000-08-26,Arsenal,5.0,3.0,H,18.0,9.0,12.0,8.0,1,...,3,2.0,1.0,3.0,1.000000,0.500000,0.50,15.500000,9.500000,9.50
3,2000-09-06,Arsenal,2.0,2.0,D,13.0,5.0,22.0,6.0,0,...,1,7.0,4.0,6.0,2.333333,1.333333,1.00,16.333333,9.333333,9.00
4,2000-09-09,Arsenal,1.0,1.0,D,18.0,11.0,13.0,10.0,0,...,1,9.0,6.0,7.0,2.250000,1.500000,0.75,15.500000,8.250000,8.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3.0,0.0,H,20.0,6.0,4.0,7.0,1,...,3,10.0,4.0,15.0,2.000000,0.800000,1.20,10.600000,3.800000,15.00
19196,2025-05-02,Wolves,0.0,1.0,H,6.0,1.0,5.0,11.0,0,...,0,11.0,3.0,15.0,2.200000,0.600000,1.60,13.600000,4.400000,13.80
19197,2025-05-10,Wolves,0.0,2.0,A,10.0,3.0,7.0,8.0,1,...,0,10.0,4.0,12.0,2.000000,0.800000,1.20,13.000000,4.200000,12.80
19198,2025-05-20,Wolves,2.0,4.0,H,12.0,3.0,10.0,8.0,0,...,0,8.0,5.0,9.0,1.600000,1.000000,0.60,10.600000,3.400000,11.20


At this point, team_history has, for each (Team, Date), features like:
    `GF_sum_last_5`, `GA_sum_last_5`, `Points_sum_last_5`,
    `GF_mean_last_5`, `GA_mean_last_5`, `GoalDiff_mean_last_5`, ...

The first few matches for each team will have NaNs
because they don't have k previous games.

In [11]:
# Pick just the columns we care about
rolling_cols = [c for c in team_history.columns if f"last_{k}" in c]
base_cols = ["Date", "Team", "is_home"]

team_feats = team_history[base_cols + rolling_cols].copy()
team_feats

,Date,Team,is_home,GF_sum_last_5,GA_sum_last_5,Points_sum_last_5,GF_mean_last_5,GA_mean_last_5,GoalDiff_mean_last_5,Shots_mean_last_5,ShotsOnTarget_mean_last_5,Corners_mean_last_5
0,2000-08-19,Arsenal,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-21,Arsenal,1,0.0,1.0,0.0,0.000000,1.000000,-1.00,14.000000,7.000000,9.00
2,2000-08-26,Arsenal,1,2.0,1.0,3.0,1.000000,0.500000,0.50,15.500000,9.500000,9.50
3,2000-09-06,Arsenal,0,7.0,4.0,6.0,2.333333,1.333333,1.00,16.333333,9.333333,9.00
4,2000-09-09,Arsenal,0,9.0,6.0,7.0,2.250000,1.500000,0.75,15.500000,8.250000,8.25
...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,1,10.0,4.0,15.0,2.000000,0.800000,1.20,10.600000,3.800000,15.00
19196,2025-05-02,Wolves,0,11.0,3.0,15.0,2.200000,0.600000,1.60,13.600000,4.400000,13.80
19197,2025-05-10,Wolves,1,10.0,4.0,12.0,2.000000,0.800000,1.20,13.000000,4.200000,12.80
19198,2025-05-20,Wolves,0,8.0,5.0,9.0,1.600000,1.000000,0.60,10.600000,3.400000,11.20


In [12]:
# Split into home and away versions
home_feats = (
    team_feats[team_feats["is_home"] == 1]
    .drop(columns=["is_home"])
    .sort_values(["Team", "Date"])
)

away_feats = (
    team_feats[team_feats["is_home"] == 0]
    .drop(columns=["is_home"])
    .sort_values(["Team", "Date"])
)

In [13]:
# Deduplicate by key: (Date, Team)
home_feats = home_feats.drop_duplicates(subset=["Date", "Team"], keep="last")
away_feats = away_feats.drop_duplicates(subset=["Date", "Team"], keep="last")

In [14]:
# Make a fresh copy of the original match-level df (with Date as datetime)
matches_with_form = train_df.copy()
matches_with_form["Date"] = pd.to_datetime(matches_with_form["Date"], dayfirst=True)

In [15]:
# Merge home form: match (Date, HomeTeam) with (home_Date, home_Team)
matches_with_form = matches_with_form.merge(
    home_feats.add_prefix("home_"),
    left_on=["Date", "HomeTeam"],
    right_on=["home_Date", "home_Team"],
    how="left"
)

# Merge away form: match (Date, AwayTeam) with (away_Date, away_Team)
matches_with_form = matches_with_form.merge(
    away_feats.add_prefix("away_"),
    left_on=["Date", "AwayTeam"],
    right_on=["away_Date", "away_Team"],
    how="left"
)

In [16]:
# Drop duplicate join keys (we still have the original Date/HomeTeam/AwayTeam)
matches_with_form = matches_with_form.drop(
    columns=["home_Date", "home_Team", "away_Date", "away_Team"]
)

In [17]:
matches_with_form["points_form_diff_last_5"] = (
    matches_with_form["home_Points_sum_last_5"]
    - matches_with_form["away_Points_sum_last_5"]
)

matches_with_form["goal_diff_form_last_5"] = (
    matches_with_form["home_GF_sum_last_5"] - matches_with_form["home_GA_sum_last_5"]
    - (matches_with_form["away_GF_sum_last_5"] - matches_with_form["away_GA_sum_last_5"])
)

In [18]:
matches_with_form

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,away_GA_sum_last_5,away_Points_sum_last_5,away_GF_mean_last_5,away_GA_mean_last_5,away_GoalDiff_mean_last_5,away_Shots_mean_last_5,away_ShotsOnTarget_mean_last_5,away_Corners_mean_last_5,points_form_diff_last_5,goal_diff_form_last_5
0,2000-08-19,Charlton,Man City,4.0,0.0,H,2.0,0.0,H,Rob Harris,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-19,Chelsea,West Ham,4.0,2.0,H,1.0,0.0,H,Graham Barber,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2000-08-19,Coventry,Middlesbrough,1.0,3.0,A,1.0,1.0,D,Barry Knight,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2000-08-19,Derby,Southampton,2.0,2.0,D,1.0,2.0,A,Andy D'Urso,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2000-08-19,Leeds,Everton,2.0,0.0,H,2.0,0.0,H,Dermot Gallagher,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9595,2025-05-25,Ipswich,West Ham,1.0,3.0,A,0.0,1.0,A,T Robinson,...,7.0,5.0,1.4,1.4,0.0,12.0,3.8,13.6,-4.0,-10.0
9596,2025-05-25,Fulham,Man City,0.0,2.0,A,0.0,1.0,A,A Madley,...,2.0,13.0,1.6,0.4,1.2,14.6,5.0,7.4,-7.0,-8.0
9597,2025-05-25,Bournemouth,Leicester,2.0,0.0,H,0.0,0.0,D,L Smith,...,6.0,7.0,1.2,1.2,0.0,9.2,3.0,13.2,-2.0,-2.0
9598,2025-05-25,Liverpool,Crystal Palace,1.0,1.0,D,0.0,1.0,A,D England,...,5.0,9.0,1.8,1.0,0.8,15.4,5.4,10.4,-2.0,-2.0


### Exponentially Weighted Moving Averages 

In [19]:
team_history = create_team_history(train_df)
team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points
0,2000-08-19,Arsenal,0.0,1.0,H,14.0,7.0,21.0,9.0,0,-1.0,0
1,2000-08-21,Arsenal,2.0,0.0,H,17.0,12.0,25.0,10.0,1,2.0,3
2,2000-08-26,Arsenal,5.0,3.0,H,18.0,9.0,12.0,8.0,1,2.0,3
3,2000-09-06,Arsenal,2.0,2.0,D,13.0,5.0,22.0,6.0,0,0.0,1
4,2000-09-09,Arsenal,1.0,1.0,D,18.0,11.0,13.0,10.0,0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3.0,0.0,H,20.0,6.0,4.0,7.0,1,3.0,3
19196,2025-05-02,Wolves,0.0,1.0,H,6.0,1.0,5.0,11.0,0,-1.0,0
19197,2025-05-10,Wolves,0.0,2.0,A,10.0,3.0,7.0,8.0,1,-2.0,0
19198,2025-05-20,Wolves,2.0,4.0,H,12.0,3.0,10.0,8.0,0,-2.0,0


In [20]:
# More recent matches are weighted more heavily.
# span controls decay speed; smaller span = faster decay.
span = 8  # hyperparameter; you can tune this

grp = team_history.groupby("Team", group_keys=False)

def ewm_shifted(col_name: str, span: int) -> pd.Series:
    """
    For each team, compute EWM over *past* matches only:
    - shift(1) so current match is NOT included (no leakage)
    - ewm(span=span) to weight recent history more.
    """
    return grp[col_name].apply(
        lambda s: s.shift(1).ewm(span=span, adjust=False).mean()
    )

# Choose which stats to smooth
ewm_source_cols = ["GF", "GA", "GoalDiff", "Points",
                   "Shots", "ShotsOnTarget", "Fouls", "Corners"]

for col in ewm_source_cols:
    team_history[f"{col}_ewm"] = ewm_shifted(col, span)

In [21]:
team_history

,Date,Team,GF,GA,FTR,Shots,ShotsOnTarget,Fouls,Corners,is_home,GoalDiff,Points,GF_ewm,GA_ewm,GoalDiff_ewm,Points_ewm,Shots_ewm,ShotsOnTarget_ewm,Fouls_ewm,Corners_ewm
0,2000-08-19,Arsenal,0.0,1.0,H,14.0,7.0,21.0,9.0,0,-1.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2000-08-21,Arsenal,2.0,0.0,H,17.0,12.0,25.0,10.0,1,2.0,3,0.000000,1.000000,-1.000000,0.000000,14.000000,7.000000,21.000000,9.000000
2,2000-08-26,Arsenal,5.0,3.0,H,18.0,9.0,12.0,8.0,1,2.0,3,0.444444,0.777778,-0.333333,0.666667,14.666667,8.111111,21.888889,9.222222
3,2000-09-06,Arsenal,2.0,2.0,D,13.0,5.0,22.0,6.0,0,0.0,1,1.456790,1.271605,0.185185,1.185185,15.407407,8.308642,19.691358,8.950617
4,2000-09-09,Arsenal,1.0,1.0,D,18.0,11.0,13.0,10.0,0,0.0,1,1.577503,1.433471,0.144033,1.144033,14.872428,7.573388,20.204390,8.294925
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19195,2025-04-26,Wolves,3.0,0.0,H,20.0,6.0,4.0,7.0,1,3.0,3,1.745443,0.947021,0.798421,2.435036,11.033330,3.938660,3.282997,14.389498
19196,2025-05-02,Wolves,0.0,1.0,H,6.0,1.0,5.0,11.0,0,-1.0,0,2.024233,0.736572,1.287661,2.560584,13.025923,4.396735,3.442331,12.747387
19197,2025-05-10,Wolves,0.0,2.0,A,10.0,3.0,7.0,8.0,1,-2.0,0,1.574404,0.795112,0.779292,1.991565,11.464607,3.641905,3.788479,12.359079
19198,2025-05-20,Wolves,2.0,4.0,H,12.0,3.0,10.0,8.0,0,-2.0,0,1.224536,1.062865,0.161671,1.548995,11.139139,3.499260,4.502151,11.390395


In [22]:
# Columns we need from team_history to join back
ewm_cols = [c for c in team_history.columns if c.endswith("_ewm")]
base_cols = ["Date", "Team", "is_home"]

team_feats = team_history[base_cols + ewm_cols].copy()

# Split into home and away feature tables
home_feats = team_feats[team_feats["is_home"] == 1].drop(columns=["is_home"])
away_feats = team_feats[team_feats["is_home"] == 0].drop(columns=["is_home"])

# Prefix columns so they stay distinct
home_feats = home_feats.add_prefix("home_")
away_feats = away_feats.add_prefix("away_")

# Fresh copy of the original match-level data
matches_with_ewm = train_df.copy()
matches_with_ewm["Date"] = pd.to_datetime(matches_with_ewm["Date"], dayfirst=True)

# Merge home EWMA features: (Date, HomeTeam) <-> (home_Date, home_Team)
matches_with_ewm = matches_with_ewm.merge(
    home_feats,
    left_on=["Date", "HomeTeam"],
    right_on=["home_Date", "home_Team"],
    how="left"
)

# Merge away EWMA features: (Date, AwayTeam) <-> (away_Date, away_Team)
matches_with_ewm = matches_with_ewm.merge(
    away_feats,
    left_on=["Date", "AwayTeam"],
    right_on=["away_Date", "away_Team"],
    how="left"
)

# Remove duplicate key columns from the merge
matches_with_ewm = matches_with_ewm.drop(
    columns=["home_Date", "home_Team", "away_Date", "away_Team"]
)

matches_with_ewm["ewm_points_diff"] = (
    matches_with_ewm["home_Points_ewm"] - matches_with_ewm["away_Points_ewm"]
)

matches_with_ewm["ewm_goal_diff_diff"] = (
    matches_with_ewm["home_GoalDiff_ewm"] - matches_with_ewm["away_GoalDiff_ewm"]
)


In [23]:
matches_with_ewm.columns

Index(['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG',
       'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF',
       'HY', 'AY', 'HR', 'AR', 'home_GF_ewm', 'home_GA_ewm',
       'home_GoalDiff_ewm', 'home_Points_ewm', 'home_Shots_ewm',
       'home_ShotsOnTarget_ewm', 'home_Fouls_ewm', 'home_Corners_ewm',
       'away_GF_ewm', 'away_GA_ewm', 'away_GoalDiff_ewm', 'away_Points_ewm',
       'away_Shots_ewm', 'away_ShotsOnTarget_ewm', 'away_Fouls_ewm',
       'away_Corners_ewm', 'ewm_points_diff', 'ewm_goal_diff_diff'],
      dtype='object')

### ELO Ratings

In [24]:
def add_elo_features(
    df: pd.DataFrame,
    base_rating: float = 1500.0,
    k_factor: float = 20.0,
    home_advantage: float = 50.0
):
    """
    Compute ELO ratings over time and attach, for each match:
      - home_elo_pre: rating of HomeTeam *before* this match
      - away_elo_pre: rating of AwayTeam *before* this match
      - elo_diff_pre: home_elo_pre - away_elo_pre

    Parameters
    ----------
    df : DataFrame with at least ['Date','HomeTeam','AwayTeam','FTR']
    base_rating : starting ELO for all teams
    k_factor : learning rate for rating updates
    home_advantage : rating points added to home team when computing expectation

    Returns
    -------
    df_with_elo : DataFrame copy with new ELO columns
    final_ratings : dict {team_name: final_elo_after_last_match}
    """

    df = df.copy()
    df = df.sort_values("Date").reset_index(drop=True)

    # Current ratings for each team
    ratings = {}

    # We’ll store pre-match ratings here
    home_elo_pre = []
    away_elo_pre = []

    def result_to_scores(ftr: str):
        """Map FTR (H/D/A) to ELO scores (S_home, S_away)."""
        if ftr == "H":
            return 1.0, 0.0
        elif ftr == "A":
            return 0.0, 1.0
        else:  # Draw
            return 0.5, 0.5

    # Iterate through matches in time order
    for _, row in df.iterrows():
        home = row["HomeTeam"]
        away = row["AwayTeam"]
        ftr = row["FTR"]

        # Get current ratings (default to base_rating the first time we see a team)
        R_h = ratings.get(home, base_rating)
        R_a = ratings.get(away, base_rating)

        # Save PRE-match ratings as features
        home_elo_pre.append(R_h)
        away_elo_pre.append(R_a)

        # --------- ELO UPDATE STEP ---------
        # Expected home score (with home advantage)
        exp_home = 1.0 / (1.0 + 10 ** (-(R_h + home_advantage - R_a) / 400.0))
        exp_away = 1.0 - exp_home

        # Actual scores from result
        S_h, S_a = result_to_scores(ftr)

        # New ratings
        R_h_new = R_h + k_factor * (S_h - exp_home)
        R_a_new = R_a + k_factor * (S_a - exp_away)

        # Store updated ratings
        ratings[home] = R_h_new
        ratings[away] = R_a_new

    # Attach pre-match ratings as columns
    df["home_elo_pre"] = home_elo_pre
    df["away_elo_pre"] = away_elo_pre
    df["elo_diff_pre"] = df["home_elo_pre"] - df["away_elo_pre"]

    return df, ratings


# Run on your training data
train_with_elo, final_elo_ratings = add_elo_features(train_df)

train_with_elo[["Date", "HomeTeam", "AwayTeam", "FTR",
         "home_elo_pre", "away_elo_pre", "elo_diff_pre"]]

,Date,HomeTeam,AwayTeam,FTR,home_elo_pre,away_elo_pre,elo_diff_pre
0,2000-08-19,Charlton,Man City,H,1500.000000,1500.000000,0.000000
1,2000-08-19,Chelsea,West Ham,H,1500.000000,1500.000000,0.000000
2,2000-08-19,Coventry,Middlesbrough,A,1500.000000,1500.000000,0.000000
3,2000-08-19,Derby,Southampton,D,1500.000000,1500.000000,0.000000
4,2000-08-19,Leeds,Everton,H,1500.000000,1500.000000,0.000000
...,...,...,...,...,...,...,...
9595,2025-05-25,Southampton,Arsenal,A,1351.780976,1772.479113,-420.698137
9596,2025-05-25,Newcastle,Everton,A,1677.011651,1577.256487,99.755164
9597,2025-05-25,Nottingham Forest,Chelsea,A,1607.715011,1673.309574,-65.594563
9598,2025-05-25,Man United,Aston Villa,H,1549.451178,1695.466458,-146.015280


### Build Features